In [1]:
from google.colab import files
uploaded = files.upload()

Saving employees.csv to employees.csv
Saving logins.txt to logins.txt
Saving orders.json to orders.json
Saving sales.csv to sales.csv


In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder\
.appName("Combined Challenge")\
.getOrCreate()

In [8]:
df_txt = spark.read.text("logins.txt")
df_txt = df_txt.withColumnRenamed("value", "name")

In [9]:
df_txt.show()

+-----+
| name|
+-----+
|Rahul|
|Sneha|
|Arjun|
|Rahul|
|Priya|
|Sneha|
|Rahul|
|Karan|
|Arjun|
|Sneha|
|Rahul|
| Amit|
|Priya|
|Karan|
|Sneha|
|Rahul|
|Meera|
|Arjun|
|Sneha|
|Rahul|
+-----+
only showing top 20 rows


In [10]:
df_txt.count()

26

In [11]:
df_txt.distinct().show()

+-----+
| name|
+-----+
|Meera|
|Sneha|
|Priya|
|Rahul|
|Arjun|
| Amit|
|Karan|
+-----+



In [12]:
df_txt.groupBy("name").count().show()

+-----+-----+
| name|count|
+-----+-----+
|Meera|    1|
|Sneha|    6|
|Priya|    3|
|Rahul|    7|
|Arjun|    4|
| Amit|    2|
|Karan|    3|
+-----+-----+



In [56]:
df_txt.groupBy("name").count()\
.orderBy("count", ascending=False).show(3)

+-----+-----+
| name|count|
+-----+-----+
|Rahul|    7|
|Sneha|    6|
|Arjun|    4|
+-----+-----+
only showing top 3 rows


In [57]:
df_txt.groupBy("name")\
.count().filter("count > 4").show()

+-----+-----+
| name|count|
+-----+-----+
|Sneha|    6|
|Rahul|    7|
+-----+-----+



In [58]:
df_txt.groupBy("name")\
.count().rdd.collectAsMap()

{'Meera': 1,
 'Sneha': 6,
 'Priya': 3,
 'Rahul': 7,
 'Arjun': 4,
 'Amit': 2,
 'Karan': 3}

In [18]:
df_emp = spark.read.csv("employees.csv", header=True, inferSchema=True)

In [19]:
df_emp

DataFrame[emp_id: int, name: string, department: string, salary: int, city: string]

In [20]:
df_emp.count()

20

In [21]:
df_emp.filter(df_emp.department == "IT").show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     1| Rahul|        IT| 70000|Hyderabad|
|     3| Arjun|        IT| 75000|  Chennai|
|     5| Karan|        IT| 50000|   Mumbai|
|     8|  Ravi|        IT| 72000|Hyderabad|
|    11| Anita|        IT| 65000|Bangalore|
|    13| Divya|        IT| 77000|Hyderabad|
|    15| Pooja|        IT| 69000|Bangalore|
|    18|Deepak|        IT| 73000|Hyderabad|
+------+------+----------+------+---------+



In [22]:
df_emp.filter(df_emp.salary > 75000).show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     4| Priya|   Finance| 80000|Hyderabad|
|     7| Meera|   Finance| 82000|Bangalore|
|    10|Vikram|   Finance| 90000|    Delhi|
|    13| Divya|        IT| 77000|Hyderabad|
|    14|Sanjay|   Finance| 85000|  Chennai|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    20| Akash|   Finance| 91000|    Delhi|
+------+------+----------+------+---------+



In [23]:
from pyspark.sql.functions import avg
df_emp.select(avg("salary")).show()

+-----------+
|avg(salary)|
+-----------+
|    71450.0|
+-----------+



In [24]:
from pyspark.sql.functions import max
df_emp.select(max("salary")).show()

+-----------+
|max(salary)|
+-----------+
|      91000|
+-----------+



In [25]:
from pyspark.sql.functions import min
df_emp.select(min("salary")).show()

+-----------+
|min(salary)|
+-----------+
|      50000|
+-----------+



In [26]:
df_emp.groupBy("department").count().show()

+----------+-----+
|department|count|
+----------+-----+
|        HR|    6|
|   Finance|    6|
|        IT|    8|
+----------+-----+



In [27]:
df_emp.groupBy("department").avg("salary").show()

+----------+------------------+
|department|       avg(salary)|
+----------+------------------+
|        HR|60333.333333333336|
|   Finance|           86000.0|
|        IT|           68875.0|
+----------+------------------+



In [28]:
df_emp.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    4|
|  Chennai|    4|
|   Mumbai|    3|
|    Delhi|    4|
|Hyderabad|    5|
+---------+-----+



In [29]:
df_emp.orderBy(df_emp.salary.desc()).show(5)

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|    20| Akash|   Finance| 91000|    Delhi|
|    10|Vikram|   Finance| 90000|    Delhi|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    14|Sanjay|   Finance| 85000|  Chennai|
|     7| Meera|   Finance| 82000|Bangalore|
+------+------+----------+------+---------+
only showing top 5 rows


In [30]:
df_emp.filter((df_emp.city == "Hyderabad") & (df_emp.salary > 70000)).show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     4| Priya|   Finance| 80000|Hyderabad|
|     8|  Ravi|        IT| 72000|Hyderabad|
|    13| Divya|        IT| 77000|Hyderabad|
|    18|Deepak|        IT| 73000|Hyderabad|
+------+------+----------+------+---------+



In [31]:
df_sales = spark.read.csv("sales.csv", header=True, inferSchema=True)
from pyspark.sql.functions import col
df_sales = df_sales.withColumn("revenue", col("quantity") * col("price"))

In [32]:
df_sales.select("sale_id", "revenue").show()

+-------+-------+
|sale_id|revenue|
+-------+-------+
|      1|  75000|
|      2|   1500|
|      3|   3000|
|      4|  12000|
|      5|  75000|
|      6|   1000|
|      7|   1500|
|      8|  75000|
|      9|   2000|
|     10|  24000|
|     11|   3000|
|     12|   1500|
|     13|  75000|
|     14|  12000|
|     15|   3000|
|     16|  75000|
|     17|   1500|
|     18|  12000|
|     19|   1500|
|     20|  75000|
+-------+-------+



In [33]:
from pyspark.sql.functions import sum
df_sales.select(sum("revenue")).show()

+------------+
|sum(revenue)|
+------------+
|      529500|
+------------+



In [34]:
df_sales.groupBy("product").sum("revenue").show()

+--------+------------+
| product|sum(revenue)|
+--------+------------+
|  Laptop|      450000|
|   Mouse|        7500|
|Keyboard|       12000|
| Monitor|       60000|
+--------+------------+



In [35]:
df_sales.groupBy("product").sum("quantity").show()

+--------+-------------+
| product|sum(quantity)|
+--------+-------------+
|  Laptop|            6|
|   Mouse|           15|
|Keyboard|            8|
| Monitor|            5|
+--------+-------------+



In [59]:
df_sales.groupBy("product").sum("quantity")\
.orderBy("sum(quantity)", ascending=False).show(1)

+-------+-------------+
|product|sum(quantity)|
+-------+-------------+
|  Mouse|           15|
+-------+-------------+
only showing top 1 row


In [60]:
df_sales.groupBy("emp_id").sum("revenue")\
.orderBy("sum(revenue)", ascending=False).show(1)

+------+------------+
|emp_id|sum(revenue)|
+------+------------+
|     1|      162000|
+------+------------+
only showing top 1 row


In [39]:
from pyspark.sql.functions import avg
df_sales.select(avg("revenue")).show()

+------------+
|avg(revenue)|
+------------+
|     26475.0|
+------------+



In [61]:
df_sales.groupBy("product").sum("revenue")\
.filter("sum(revenue) > 100000").show()

+-------+------------+
|product|sum(revenue)|
+-------+------------+
| Laptop|      450000|
+-------+------------+



In [63]:
df_json = spark.read.option("multiline", "true")\
.json("orders.json")
from pyspark.sql.functions import explode
df_orders = df_json\
.select(explode("orders").alias("data")).select("data.*")

In [64]:
df_orders

DataFrame[amount: bigint, city: string, customer: string, order_id: bigint, product: string]

In [43]:
df_orders.show()

+------+---------+--------+--------+--------+
|amount|     city|customer|order_id| product|
+------+---------+--------+--------+--------+
| 75000|Hyderabad|   Rahul|       1|  Laptop|
|  1500|Bangalore|   Sneha|       2|   Mouse|
|  3000|  Chennai|   Arjun|       3|Keyboard|
| 75000|Hyderabad|   Priya|       4|  Laptop|
| 12000|   Mumbai|   Karan|       5| Monitor|
|  1000|Hyderabad|   Rahul|       6|   Mouse|
| 75000|Bangalore|   Sneha|       7|  Laptop|
|  3000|  Chennai|   Arjun|       8|Keyboard|
|  2000|Hyderabad|   Priya|       9|   Mouse|
| 12000|Hyderabad|   Rahul|      10| Monitor|
+------+---------+--------+--------+--------+



In [44]:
df_orders.count()

10

In [45]:
from pyspark.sql.functions import sum
df_orders.select(sum("amount")).show()

+-----------+
|sum(amount)|
+-----------+
|     259500|
+-----------+



In [46]:
df_orders.groupBy("customer").sum("amount").show()

+--------+-----------+
|customer|sum(amount)|
+--------+-----------+
|   Sneha|      76500|
|   Priya|      77000|
|   Rahul|      88000|
|   Arjun|       6000|
|   Karan|      12000|
+--------+-----------+



In [65]:
df_orders.groupBy("customer").sum("amount")\
.orderBy("sum(amount)", ascending=False).show(1)

+--------+-----------+
|customer|sum(amount)|
+--------+-----------+
|   Rahul|      88000|
+--------+-----------+
only showing top 1 row


In [48]:
df_orders.groupBy("product").sum("amount").show()

+--------+-----------+
| product|sum(amount)|
+--------+-----------+
|  Laptop|     225000|
|   Mouse|       4500|
|Keyboard|       6000|
| Monitor|      24000|
+--------+-----------+



In [49]:
df_orders.filter(df_orders.city == "Hyderabad").show()

+------+---------+--------+--------+-------+
|amount|     city|customer|order_id|product|
+------+---------+--------+--------+-------+
| 75000|Hyderabad|   Rahul|       1| Laptop|
| 75000|Hyderabad|   Priya|       4| Laptop|
|  1000|Hyderabad|   Rahul|       6|  Mouse|
|  2000|Hyderabad|   Priya|       9|  Mouse|
| 12000|Hyderabad|   Rahul|      10|Monitor|
+------+---------+--------+--------+-------+



In [50]:
df_orders.filter(df_orders.amount > 10000).show()

+------+---------+--------+--------+-------+
|amount|     city|customer|order_id|product|
+------+---------+--------+--------+-------+
| 75000|Hyderabad|   Rahul|       1| Laptop|
| 75000|Hyderabad|   Priya|       4| Laptop|
| 12000|   Mumbai|   Karan|       5|Monitor|
| 75000|Bangalore|   Sneha|       7| Laptop|
| 12000|Hyderabad|   Rahul|      10|Monitor|
+------+---------+--------+--------+-------+



In [51]:
df_orders.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    2|
|   Mumbai|    1|
|Hyderabad|    5|
+---------+-----+



In [52]:
df_join = df_sales.join(df_emp, on="emp_id")
df_join

DataFrame[emp_id: int, sale_id: int, product: string, quantity: int, price: int, revenue: int, name: string, department: string, salary: int, city: string]

In [53]:
df_join.groupBy("name").sum("revenue").show()

+------+------------+
|  name|sum(revenue)|
+------+------------+
| Kunal|        1500|
| Sonal|       75000|
| Divya|       75000|
|  Ravi|        3000|
|Sanjay|        1500|
| Meera|       24000|
| Sneha|        1500|
| Priya|       75000|
|Vikram|       75000|
| Rahul|      162000|
| Anita|       12000|
| Manoj|        3000|
| Pooja|       12000|
| Arjun|        4000|
|  Amit|        2000|
|  Neha|        1500|
| Karan|        1500|
+------+------------+



In [66]:
df_join.groupBy("name").sum("revenue")\
.orderBy("sum(revenue)", ascending=False).show(5)

+------+------------+
|  name|sum(revenue)|
+------+------------+
| Rahul|      162000|
| Priya|       75000|
| Divya|       75000|
| Sonal|       75000|
|Vikram|       75000|
+------+------------+
only showing top 5 rows


In [67]:
df_join.groupBy("department").sum("revenue")\
.orderBy("sum(revenue)", ascending=False).show(1)

+----------+------------+
|department|sum(revenue)|
+----------+------------+
|        IT|      269500|
+----------+------------+
only showing top 1 row
